## Блок импорта всех необходимых библиотек  

Этот блок импортирует ключевые библиотеки, используемые для работы с геоданными, API-запросами и вычислениями.  

**Действия и назначение библиотек:**  
- `os` — взаимодействие с файловой системой (работа с путями, проверка существования файлов и директорий)  
- `math` — выполнение базовых математических операций (округление, вычисление расстояний, тригонометрические функции)  
- `time` — управление временными задержками, измерение времени выполнения операций  
- `random` — генерация случайных чисел, полезная при отладке или создании случайных задержек при API-запросах  
- `json` — чтение и запись данных в формате JSON (используется при работе с ответами API)  
- `requests` — отправка HTTP-запросов и получение ответов от внешних сервисов (например, геокодеров)  
- `pandas` — обработка и анализ табличных данных (загрузка, фильтрация, объединение, сохранение)  
- `tqdm` — визуализация прогресса выполнения циклов с индикатором загрузки  
- `geopy.distance` (`geodesic`) — вычисление географического расстояния между двумя координатами (широта и долгота)  


In [4]:
# Импорт необходимых библиотек
import os
import math
import time
import random
import json

import requests
import pandas as pd
from tqdm import tqdm
from geopy.distance import geodesic

## Блок загрузки исходных данных из GitHub  

Данный блок отвечает за загрузку очищенных данных, необходимых для дальнейшего парсинга данных для банкоматов. Источник данных — репозиторий GitHub, откуда CSV-файл подгружается напрямую через ссылку на его **RAW-версию**.  

In [3]:
# Загрузка исходных данных из Github

# Ссылка на raw-версию файла из репозитория GitHub
clean_data = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/geo_data_cleaned.csv"

# Чтение файла
clean_data = pd.read_csv(clean_data)

display(clean_data)

,id,atm_group,address,geo_address,geo_lon,geo_lat,country,province,area,locality,street,house,target,isTrain
0,1.0,32.0,"MARATA, 17 S.PETERBURG","Россия, Санкт-Петербург, улица Марата, 17",30.353382,59.928916,Россия,Санкт-Петербург,NaN,Санкт-Петербург,улица Марата,17,NaN,False
1,2.0,32.0,"TYUSHINA, 11 ST.PETERSBURG","Россия, Санкт-Петербург, улица Тюшина, 11",30.346887,59.917619,Россия,Санкт-Петербург,NaN,Санкт-Петербург,улица Тюшина,11,NaN,False
2,3.0,1022.0,ABB 5 M.DZHALILYA N.SHIGALEEVO,"Россия, Республика Татарстан (Татарстан), Пест...",49.443201,55.807779,Россия,Республика Татарстан (Татарстан),Шигалеевское сельское поселение,село Новое Шигалеево,улица Мусы Джалиля,5,NaN,False
3,4.0,496.5,PUSHKINA 15 ELISTA,"Россия, Республика Калмыкия, Элиста, улица А.С...",44.270019,46.309498,Россия,Республика Калмыкия,городской округ Элиста,Элиста,улица А.С. Пушкина,15,NaN,False
4,5.0,496.5,BUDENNOGO 7A ELISTA,"Россия, Республика Калмыкия, Элиста, улица С.М...",44.260605,46.318231,Россия,Республика Калмыкия,городской округ Элиста,Элиста,улица С.М. Будённого,7А,0.019958,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8677,8807.0,496.5,OKTYABRSKAYA 18A IKI-BURUL,"Россия, Республика Калмыкия, посёлок Ики-Бурул...",44.647851,45.826910,Россия,Республика Калмыкия,Ики-Бурульский район,посёлок Ики-Бурул,Октябрьская улица,16,-0.090848,True
8678,8808.0,496.5,REVOLUTIONNAYA 1 LAGAN,"Россия, Республика Калмыкия, Лагань, Революцио...",47.377247,45.400131,Россия,Республика Калмыкия,Лаганское городское муниципальное образование,Лагань,Революционная улица,1,NaN,False
8679,8809.0,496.5,"ZHIGULSKOGO 1""Z"" LAGAN","Россия, Республика Калмыкия, Лагань, улица Жиг...",47.361688,45.389283,Россия,Республика Калмыкия,Лаганское городское муниципальное образование,Лагань,улица Жигульского,1,0.036916,True
8680,8810.0,496.5,GORODOVIKOVA 3A ELISTA,"Россия, Республика Калмыкия, Элиста, улица Б. ...",44.266498,46.306946,Россия,Республика Калмыкия,городской округ Элиста,Элиста,улица Б. Городовикова,3А,-0.135684,True


## Блок парсинга данных №1
Блок выполняет поиск ближайших объектов инфраструктуры (ТЦ, магазины, аптеки, банки) вокруг каждого банкомата с помощью **Overpass API (OpenStreetMap)**.  
Для каждого банкомата определяется ближайший объект и количество объектов в радиусах **100 / 300 / 500 м**, а также оценивается конкурентное окружение (банкоматы и отделения других банков).

### Основная логика
1. Для каждой точки банкомата выполняется запрос к Overpass API по заданным фильтрам (`OSM_FILTERS`).
2. Для найденных объектов рассчитываются расстояния по формуле гаверсинусов.
3. Определяется ближайший объект и подсчитывается число объектов в радиусах 100 / 300 / 500 м.
4. Для категории банков дополнительно считается количество конкурентов (`operator != наш банк`).
5. Результаты сохраняются в CSV с промежуточными сохранениями каждые 300 итераций.

### Ключевые функции

**`haversine_m(lat1, lon1, lat2, lon2)`** —  
вычисляет расстояние между двумя точками по поверхности Земли (в метрах).

**`fetch_osm_objects(lat, lon, filters)`** —  
делает запрос к Overpass API и возвращает список найденных объектов по заданным координатам и фильтрам OSM.

Результатом работы блока является таблица с признаками:
`nearest_<category>_dist_m`, `count_<category>_<radius>m`, `competitor_count_total`.


In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    """
    Считает расстояние между двумя точками по формуле гаверсинусов (в метрах).
    """
    R = 6371000  # радиус Земли в метрах
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [ ]:
def fetch_osm_objects(lat, lon, filters):
    """
    Делает запрос к Overpass API, чтобы получить целевые объекты вокруг заданных координат.
    Возвращает список элементов (в формате JSON).
    """
    SEARCH_RADIUS = 700 # радиус поиска в метрах
    url = "https://overpass-api.de/api/interpreter" # API для запросов к OSM
    
    parts = []
    # Поиск по кокретным "точкам" или "площадям" (как ТЦ, больницы и тд)
    for f in filters:
        parts.append(f'node[{f}](around:{SEARCH_RADIUS},{lat},{lon});')
        parts.append(f'way[{f}](around:{SEARCH_RADIUS},{lat},{lon});')

    # Собираем запрос в синтаксисе Overpass QL
    query = f"[out:json][timeout:60];({''.join(parts)});out center;"

    try:
        # Отправляем запрос
        r = requests.get(url, params={"data": query}, timeout=60)
        if r.status_code == 200:
            return r.json().get("elements", [])
        else:
            print(f"️Overpass {r.status_code}: {r.text[:120]}")
            return []
    except Exception as e:
        print(f"Ошибка запроса Overpass: {e}")
        return []

In [ ]:
# Пути к входному и выходному файлам
OUTPUT_PATH = "data/POI_1.csv"

# Загружаем датасет
df = clean_data.copy()

# Проверяем, есть ли уже сохранённый файл (для докачки)
if os.path.exists(OUTPUT_PATH):
    processed_df = pd.read_csv(OUTPUT_PATH)
    processed_ids = set(processed_df.get("id", []))
    print(f"Найден сохранённый файл: {OUTPUT_PATH} ({len(processed_df)} строк)")
else:
    processed_df = pd.DataFrame()
    processed_ids = set()
    print("Сохранённый файл не найден, начинаем с нуля.")


# Фильтры OSM для разных типов объектов
OSM_FILTERS = {
    "malls": ['shop="mall"'], # Торговые центры
    "supermarkets": ['shop="supermarket"', 'shop="alcohol"'], # Супермаркеты и алкомаркеты
    "pharmacies_hospitals": ['amenity="pharmacy"', 'amenity="hospital"', 'amenity="clinic"'], # Аптеки и клиники
    "banks_atms": ['amenity="bank"', 'amenity="atm"'], # Отделения и банкоматы
}

RADII = [100, 300, 500] # Радиусы, по которым считаем количество целевых объектов

results = []

# Основной цикл по всем банкоматам
for i, row in df.iterrows():
    lat, lon = float(row["geo_lat"]), float(row["geo_lon"])
    atm_id = row.get("id", i)

    print(f"\nБанкомат {i+1}/{len(df)}: ({lat:.5f}, {lon:.5f})")

    # Словарь для хранения результатов по данному банкомату
    row_result = {"atm_id": atm_id, "atm_lat": lat, "atm_lon": lon}

    # Обрабатываем каждый тип объектов (ТЦ, аптеки, отделения и т.п.)
    for key, filters in OSM_FILTERS.items():
        elements = fetch_osm_objects(lat, lon, filters)

        # Если ничего не найдено в необходимом радиусе заполняем нулями и пустыми значениями
        if not elements:
            for R in RADII:
                row_result[f"count_{key}_{R}m"] = 0
            row_result[f"nearest_{key}_name"] = ""
            row_result[f"nearest_{key}_dist_m"] = None
            continue

        # Если что-то нашли в нужном радиусе - считаем расстояния до всех найденных объектов
        dists = []
        for el in elements:
            tags = el.get("tags", {}) or {}
            name = tags.get("name", "")
            operator = tags.get("operator", "")
            el_lat = el.get("lat") or (el.get("center") or {}).get("lat")
            el_lon = el.get("lon") or (el.get("center") or {}).get("lon")
            if el_lat is None or el_lon is None:
                continue
            dist = haversine_m(lat, lon, el_lat, el_lon)
            dists.append((name, operator, dist))

        # Если что-то нашли в нужном радиусе — записываем ближайший объект и считаем объекты по радиусам (100 / 300 / 500)
        if dists:
            nearest = min(dists, key=lambda x: x[2])
            row_result[f"nearest_{key}_name"] = nearest[0]
            row_result[f"nearest_{key}_dist_m"] = round(nearest[2], 1)
            for R in RADII:
                row_result[f"count_{key}_{R}m"] = sum(d <= R for _, _, d in dists)
        else:
            # Если не удалось посчитать расстояния — заполняем нулями
            for R in RADII:
                row_result[f"count_{key}_{R}m"] = 0
            row_result[f"nearest_{key}_name"] = ""
            row_result[f"nearest_{key}_dist_m"] = None

        # Дополнительно: считаем конкурентов среди банков и банкоматов
        if key == "banks_atms":
            competitors = [(n, o, d) for n, o, d in dists if "росбанк" not in f"{n.lower()} {o.lower()}"]
            row_result["competitor_count_total"] = len(competitors)

        # Случайная пауза между запросами
        time.sleep(random.uniform(2.5, 5.0))

    results.append(row_result)

    # Каждые 300 банкоматов итераций делаем промежуточное сохранение
    if len(results) % 300 == 0:
        temp_df = pd.DataFrame(results)
        combined_df = pd.concat([processed_df, temp_df], ignore_index=True)
        combined_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
        print(f"Промежуточное сохранение: {len(combined_df)} строк → {OUTPUT_PATH}")
        processed_df = combined_df.copy()
        results = []

# Финальное сохранение результата
final_df = pd.concat([processed_df, pd.DataFrame(results)], ignore_index=True)
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"\nГотово: обработано {len(final_df)} банкоматов")
print(f"Файл сохранён: {OUTPUT_PATH}")

## Блок парсинга данных №2
Блок выполняет поиск ближайших объектов инфраструктуры (ТЦ, магазины, аптеки, банки) вокруг каждого банкомата с помощью **Overpass API (OpenStreetMap)**.  
Для каждого банкомата определяется ближайший объект и количество объектов в радиусах **100 / 300 / 500 м**, а также оценивается конкурентное окружение (банкоматы и отделения других банков).

### Основная логика
1. Для каждой точки банкомата выполняется запрос к Overpass API по заданным фильтрам (`OSM_FILTERS`).
2. Для найденных объектов рассчитываются расстояния по формуле гаверсинусов.
3. Определяется ближайший объект и подсчитывается число объектов в радиусах 100 / 300 / 500 м.
4. Для категории банков дополнительно считается количество конкурентов (`operator != наш банк`).
5. Результаты сохраняются в CSV с промежуточными сохранениями каждые 300 итераций.

### Ключевые функции

**`haversine_m(lat1, lon1, lat2, lon2)`** —  
вычисляет расстояние между двумя точками по поверхности Земли (в метрах).

**`fetch_osm_objects(lat, lon, filters)`** —  
делает запрос к Overpass API и возвращает список найденных объектов по заданным координатам и фильтрам OSM.

Результатом работы блока является таблица с признаками:
`nearest_<category>_dist_m`, `count_<category>_<radius>m`, `competitor_count_total`.


### 1. Ближайшие объекты (100/300/500 м):


*   остановки общественного транспорта
*   кафе, рестораны, кофейни
*   парковк

In [ ]:
def distance_m(lat1, lon1, lat2, lon2):
  """
  Функция для расчета расстояния между обрабатываемым банкоматом и близжашим интересующим нас объектом с помощью библиотеки geopy.
  """
  return geodesic((lat1, lon1), (lat2, lon2)).meters

In [ ]:
def osm_queries(lat, lon, filters):
    """
    Выполняет запрос к Overpass API (OpenStreetMap) для поиска объектов в заданном радиусе.

    Функция формирует и отправляет запрос в Overpass API на основе переданных фильтров OSM
    и координат точки. Возвращает список найденных объектов, соответствующих
    фильтрам в пределах заданного радиуса.

    Алгоритм работы:
    ----------------
    1. Для каждого фильтра из списка `filters` формируется подзапрос вида:
       - node[filter](around:R, lat, lon);
       - way[filter](around:R, lat, lon);
       где R — радиус поиска (`NEAREST_RADIUS`), lat/lon — координаты точки.
    2. Все подзапросы объединяются в один запрос к Overpass API.
    3. Отправляется HTTP-запрос методом GET к `url` с параметром `data=<query>`.
    4. В случае успеха (код 200) возвращаются элементы (`elements`) из JSON-ответа.
    5. При ошибках:
       - 504 (таймаут) — выполняется повтор через 5 секунд, максимум 3 попытки;
       - другие коды ошибок или исключения — выводится сообщение и возвращается пустой список.

    Параметры:
    ----------
    lat : float
        Широта точки поиска (в градусах).
    lon : float
        Долгота точки поиска (в градусах).
    filters : list[str]
        Список фильтров OSM, например ["amenity=bank", "shop=supermarket"].
        Каждый фильтр подставляется в запрос как [filter].
    """
    url = "https://overpass-api.de/api/interpreter"
    NEAREST_RADIUS = 1000

    objects = []
    for f in filters:
        objects.append(f"node[{f}](around:{NEAREST_RADIUS},{lat},{lon});")
        objects.append(f"way[{f}](around:{NEAREST_RADIUS},{lat},{lon});")
    query = f"[out:json][timeout:60];({''.join(objects)});out center;"

    for _ in range(3):
        try:
            r = requests.get(url, params={"data": query}, timeout=60)
            if r.status_code == 200:
                return r.json().get("elements", [])
            elif r.status_code == 504:
                print("Ошибка 504 (таймаут), повтор через 5 секунд")
                time.sleep(5)
                continue
            else:
                print(f"Ошибка {r.status_code}")
                return []
        except Exception as e:
            print(f"Ошибка запроса: {e}")
            time.sleep(3)
    return []

In [ ]:
def parsing(df, dataset_name, out_file, start_from):
    """
    Основная функция парсинга данных о ближайших объектах для набора банкоматов.

    Функция обрабатывает переданный DataFrame с координатами банкоматов, выполняет поиск
    ближайших объектов различных типов (кафе, рестораны и т.д.) с помощью функции
    'osm_queries()', вычисляет расстояния до близжайшего объекта категории с помощью
    функции 'distance_m(lat1, lon1, lat2, lon2)'и сохраняет результаты в CSV-файл.

    Алгоритм работы:
    1. Проверяет наличие выходного файла 'out_file'.
       Если файл найден, то начинается с первого необработанного банкомата.
    2. Для каждой новой записи в 'df':
       - Извлекает координаты (широту, долготу) банкомата.
       - Для каждого фильтра из словаря 'OSM_FILTERS' выполняет запрос к OSM.
       - Если объекты найдены:
           * вычисляет расстояние до каждого;
           * определяет ближайший объект (минимальное расстояние);
           * считает количество объектов в пределах заданных радиусов ('RADII');
           * добавляет результаты в 'row_result'.
       - Если объекты не найдены — заполняет поля нулями или текстом "Нет объектов".
    3. После каждого обработанного банкомата выполняет автосохранение (каждые 'SAVE_EVERY' записей).
    4. Сохраняет финальный DataFrame в 'out_file'.

    Параметры:
    ----------
    df : pandas.DataFrame
        Исходный DataFrame, содержащий хотя бы колонки 'geo_lat', 'geo_lon' и 'id'.
    dataset_name : str
        Название набора данных — используется для логов и вывода.
    out_file : str
        Путь к CSV-файлу для сохранения результатов.
    start_from : int
        Индекс, с которого начинать обработку (например, если нужно пропустить первые N строк).

    Глобальные зависимости:
    ------------------------
    - OSM_FILTERS : dict — словарь с типами объектов и их фильтрами для OSM-запросов.
    - RADII : list[int] — список радиусов для подсчёта количества близжайших объектов.
    - SAVE_EVERY : int — период автосохранения.
    - SLEEP : float — базовая задержка между запросами (для избежания блокировки API).
    - Функции: osm_queries(), distance_m().

    Вывод:
    ------
    CSV-файл 'out_file' с добавленными колонками:
    - nearest_<тип>_name — название ближайшего объекта.
    - nearest_<тип>_dist_m — расстояние до ближайшего объекта (м).
    - count_<тип>_<R>m — количество объектов данного типа в пределах радиуса R метров.
    """
    #Фильтры для поиска объектов с помощью OpenStreetMap (кафе и кофейни, рестораны, остановки общественного транспорта, парковки)
    OSM_FILTERS = {
        "cafes": ['amenity="cafe"', 'amenity="coffee_shop"'],
        "restaurants": ['amenity="restaurant"', 'amenity="fast_food"'],
        "public_transport": ['public_transport="stop_position"', 'highway="bus_stop"'],
        "parking": ['amenity="parking"']
    }

    RADII = [100, 300, 500]
    SLEEP = 2.0
    SAVE_EVERY = 1

    if os.path.exists(out_file):
        processed_df = pd.read_csv(out_file)
        processed_ids = set(processed_df["id"])
        print(f"Найден файл {out_file}, продолжаем с {len(processed_ids)} записей")
    else:
        processed_df = pd.DataFrame()
        processed_ids = set()
        print(f"Файл {out_file} не найден, начинаем с нуля")

    results = []
    start_index = max(len(processed_ids), start_from)

    for i, row in df.iloc[start_index:].iterrows():
        lat, long = float(row["geo_lat"]), float(row["geo_lon"])
        atm_id = row.get("id", i)
        print(f"\n[{dataset_name.upper()}] Банкомат {i+1}/{len(df)}: ({lat:.5f}, {long:.5f})")

        row_result = {"id": atm_id, "lat": lat, "lon": long}

        for key, filters in OSM_FILTERS.items():
            objects = osm_queries(lat, long, filters)

            if not objects:
                print(f"Нет объектов {key}")
                for R in RADII:
                    row_result[f"count_{key}_{R}m"] = 0
                row_result[f"nearest_{key}_name"] = "Нет объектов"
                row_result[f"nearest_{key}_dist_m"] = "Нет объектов"
                continue

            dists = []
            for obj in objects:
                tags = obj.get("tags", {})
                name = tags.get("name", "")
                obj_lat = obj.get("lat") or obj.get("center", {}).get("lat")
                obj_lon = obj.get("lon") or obj.get("center", {}).get("lon")
                if not obj_lat or not obj_lon:
                    continue
                dist = distance_m(lat, long, obj_lat, obj_lon)
                dists.append((name, dist))

            if not dists:
                continue

            nearest = min(dists, key=lambda x: x[1])
            nearest_name = nearest[0] if nearest[0] else "Не удалось узнать название"
            row_result[f"nearest_{key}_name"] = nearest_name
            row_result[f"nearest_{key}_dist_m"] = round(nearest[1], 1)

            for R in RADII:
                row_result[f"count_{key}_{R}m"] = sum(d <= R for _, d in dists)

            time.sleep(SLEEP + random.uniform(0.2, 0.5))

        results.append(row_result)

        # Автосохранение после обработки каждого банкомата
        if len(results) % SAVE_EVERY == 0:
            temp_df = pd.DataFrame(results)
            combined_df = pd.concat([processed_df, temp_df], ignore_index=True)
            combined_df.to_csv(out_file, index=False, encoding="utf-8")
            print(f"Промежуточное сохранение: {len(combined_df)} строк")
            processed_df = combined_df.copy()
            results = []

    final_df = pd.concat([processed_df, pd.DataFrame(results)], ignore_index=True)
    final_df.to_csv(out_file, index=False, encoding="utf-8")
    print(f"\nОбработано {len(final_df)} банкоматов ({dataset_name})")
    print(f"Файл сохранён: {out_file}")

In [ ]:
# Путь для сохранения датасета
OUTPUT_PATH = "data/POI_2.csv"

#Запуск парсинга
parsing(clean_data.copy(), "Clean data", OUTPUT_PATH, start_from=0)

### 2. Информация о наших банкоматах


*   Режим работы
*   Доступные операции

Информация о банкоматах собрана из сервиса Яндекс Карты из карточек банкоматов, представленных в датасете, с помощью парсера веб-страниц. Ниже представлена таблица с сырыми данными:

In [ ]:
raw_data = pd.read_csv("/content/drive/MyDrive/raw_data_ATMs.csv")
display(raw_data.head())

,id,lat,long,atm_group,bank_name,Bank_Title,Is_24_7,Is_Closed,Snippet_Rating,Working_hours,YaMaps_Rating,Operations,Валюта банкомата,Валюта приема,Виды операций,Бесконтактные технологии,QR-коды,Операции со счетами и картами
0,1.0,59.921547,30.345486,32.0,УРАЛСИБ БАНК,Банк Уралсиб,False,False,нет оценки,нет информации,нет оценки,нет информации,нет информации,нет информации,нет информации,False,False,нет информации
1,2.0,59.917619,30.346887,32.0,УРАЛСИБ БАНК,Банк Уралсиб,False,False,нет оценки,нет информации,нет оценки,нет информации,нет информации,нет информации,нет информации,False,False,нет информации
2,3.0,55.807779,49.443201,1022.0,АК Барс,Ак Барс Банк,False,False,нет оценки,нет информации,нет оценки,нет информации,нет информации,нет информации,нет информации,False,False,нет информации
3,4.0,46.309498,44.270019,496.5,Россельхозбанк,Россельхозбанк,False,False,нет оценки,нет информации,нет оценки,Валюта банкомата:российский рубль; Валюта банк...,Россельхозбанк Обзор Фото 9 Отзывы Филиалы Осо...,нет информации,нет информации,False,False,Россельхозбанк Обзор Фото 9 Отзывы Филиалы Осо...
4,9.0,51.550042,46.000390,496.5,Россельхозбанк,Россельхозбанк,False,False,нет оценки,нет информации,нет оценки,Дополнительные возможности:оплата коммерческог...,Россельхозбанк Обзор Фото 13 Отзывы 1 Филиалы ...,Россельхозбанк Обзор Фото 13 Отзывы 1 Филиалы ...,Россельхозбанк Обзор Фото 13 Отзывы 1 Филиалы ...,True,True,Россельхозбанк Обзор Фото 13 Отзывы 1 Филиалы ...


Сырые данные из карточки обрабатывались с помощью извлечения ключевых слов
и текста на вкладке "Особенности" в Яндекс Картах.

In [ ]:
cols_to_check = [
    "Operations",
    "Валюта банкомата",
    "Валюта приема",
    "Виды операций",
    "Операции со счетами и картами"
]

# Объединяем текст из всех колонок в одну строку для каждой строки
raw_data["Operations"] = raw_data[cols_to_check].astype(str).agg(" ".join, axis=1).str.lower()


In [ ]:
# Извлечение информации о валютных операциях
raw_data["USD"] = raw_data["Operations"].str.contains(r"\b(usd|доллар)\b", case=False, na=False)
raw_data["EUR"] = raw_data["Operations"].str.contains(r"\b(eur|евро)\b", case=False, na=False)

#Извлечение информации об операциях с наличными
raw_data["Cash_In"] = raw_data["Operations"].str.contains(r"\b(приём наличных|внесение наличных)\b", case=False, na=False)
raw_data["Cash_Out"] = raw_data["Operations"].str.contains(r"\b(выдача наличных|выдача наличности)\b", case=False, na=False)

#Извлечение информации о безналичной оплате
raw_data["Non-cash_pay"] = raw_data["Operations"].str.contains(r"\b(безналичная оплата)\b", case=False, na=False)

#Извлечение информации о возможности получить выписку по счету
raw_data["Account_statement"] = raw_data["Operations"].str.contains(r"\b(выписка по счёту)\b", case=False, na=False)

raw_data = raw_data.rename(columns = {'Бесконтактные технологии':'Contactless_technologies', 'QR-коды':'QR-codes'})

#Доступность помещения на инвалидной коляске:доступно
raw_data["Acсess_for_disabled"] = raw_data["Operations"].str.contains(r"\b(Доступность помещения на инвалидной коляске:доступно|Доступность входа на инвалидной коляске:доступно)\b", case=False, na=False)

#Перевод с карты на карту
raw_data["Transfer_p2p"] = raw_data["Operations"].str.contains(r"\b(Перевод с карты на карту)\b", case=False, na=False)

#Перевод между счетами
raw_data["Transfer_a2a"] = raw_data["Operations"].str.contains(r"\b(перевод между счетами)\b", case=False, na=False)

#Оплата по кредиту
raw_data["Loan_pay"] = raw_data["Operations"].str.contains(r"\b(оплата кредита)\b", case=False, na=False)

/tmp/ipython-input-3748585128.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  raw_data["USD"] = raw_data["Operations"].str.contains(r"\b(usd|доллар)\b", case=False, na=False)
/tmp/ipython-input-3748585128.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  raw_data["EUR"] = raw_data["Operations"].str.contains(r"\b(eur|евро)\b", case=False, na=False)


In [ ]:
final = raw_data.drop(columns = ['long','lat','Bank_Title', 'Operations', 'Валюта банкомата', 'Валюта приема', 'Виды операций', 'Операции со счетами и картами', 'bank_name', 'atm_group', 'Snippet_Rating', 'Working_hours', 'YaMaps_Rating', 'Is_Closed'])
display(final)


#### **Используемые столбцы**

- `Is_24_7` - True, если в сниппете банкомата в Яндекс Картах указано, что он работает **Круглосуточно**, False в противном случае.
- `Contactless_technologies` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **Бесконтактные технологии**, False в противном случае.
- `QR-codes` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **QR - коды**, False в противном случае.
- `USD` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает операции с **долларами США**, False в противном случае.
- `EUR` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает операции с **евро**, False в противном случае.
- `Cash_In` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **Внесение наличных**, False в противном случае.
- `Cash_Out` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **Снятие наличных**, False в противном случае.
- `Non_cash_pay` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **Безналичные операции**, False в противном случае.
- `Account_statement` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **Выдачу выписки по счету**, False в противном случае.
- `Acсess_for_disabled` - True, если в сниппете банкомата в Яндекс Картах указано, что он располагается **Доступно для инвалидов**, False в противном случае.
- `Transfer_p2p` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **перевод с карты на карту**, False в противном случае.
- `Transfer_a2a` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает **переводы между счетами**, False в противном случае.
- `Loan_pay` - True, если в сниппете банкомата в Яндекс Картах указано, что он поддерживает операции **оплаты по кредиту**, False в противном случае.

In [ ]:
# Ссылка на raw-версию таблицы с уже обработанными данными из репозитория GitHub
df_maps = pd.read_csv("https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/ATM_info.csv")
df_maps.head()

,id,Is_24_7,Contactless_technologies,QR-codes,USD,EUR,Cash_In,Cash_Out,Non-cash_pay,Account_statement,Acсess_for_disabled,Transfer_p2p,Transfer_a2a,Loan_pay
0,2.0,False,False,False,False,False,False,False,False,False,False,False,False,False
1,3.0,False,False,False,False,False,False,False,False,False,False,False,False,False
2,4.0,False,False,False,False,False,True,True,False,True,True,True,False,False
3,9.0,False,True,True,False,False,True,True,True,True,True,True,False,False
4,11.0,False,True,True,False,False,True,True,True,True,False,True,False,False


## Блок функций для парсинга инфраструктуры из OpenStreetMap (OSM)

Данный блок предназначен для **автоматического сбора данных о ближайших объектах инфраструктуры** (точки интереса, POI) вокруг банкоматов.  
Функции взаимодействуют с **Overpass API** — сервисом, предоставляющим доступ к данным OpenStreetMap через язык запросов **OverpassQL**.  
Результаты сохраняются в CSV-файл `data/poi_3.csv`, который используется для дальнейшего обогащения и анализа признаков — например, количества школ, метро, отделений почты и т.д. в радиусе 300 метров от каждого банкомата.  

---

### **Функция `safe_overpass_request(query, retries=5)`**

Выполняет **безопасный запрос к Overpass API** с поддержкой повторных попыток, задержек и переключения между зеркалами API при сетевых сбоях.  
Используется в цикле парсинга, чтобы гарантировать устойчивое выполнение большого количества запросов.

**Основные возможности:**
- Автоматические повторы при ошибках соединения или превышении лимита запросов (`HTTP 429`, `HTTP 504`);  
- Паузы между попытками с экспоненциальным увеличением ожидания;  
- Переключение между несколькими зеркалами Overpass API для повышения надёжности;  
- Возврат JSON-ответа или `None` в случае неудачи.

**Используется для:** безопасного получения данных о ближайших POI по сформированным OverpassQL-запросам.

---

### **Функция `process_osm_dataset(df)`**

Обрабатывает входной `DataFrame` с банкоматами, выполняя поиск **объектов инфраструктуры в радиусе 300 метров** от каждой точки.  
Результаты парсинга сохраняются в `data/poi_3.csv` и дополняются при каждом новом запуске (поддерживается возобновление с места остановки через `poi_3_progress.json`).

**Основные этапы работы:**
1. **Чтение прогресса** — восстановление индекса последней успешно обработанной строки (при повторном запуске).  
2. **Формирование OverpassQL-запроса** — поиск инфраструктурных объектов (школы, университеты, метро, почтовые отделения и др.) в радиусе 300 м от координат банкомата.  
3. **Отправка запроса** через `safe_overpass_request()` и обработка полученного JSON-ответа.  
4. **Сохранение промежуточных результатов** каждые 100 объектов в CSV, чтобы не терять данные при обрыве.  
5. **Логирование и задержки** — фиксируется прогресс выполнения, добавляются случайные паузы между запросами для обхода лимитов API.  

**На выходе:**  
Формируется CSV-файл `data/poi_3.csv`, где каждая строка соответствует одному объекту, найденному рядом с банкоматом, и содержит поля:
- `atm_id` — ID банкомата;  
- `object_id`, `object_type` — идентификатор и тип объекта в OSM (`node`, `way`);  
- `object_lat`, `object_lon` — координаты объекта;  
- `object_name` — название объекта;  
- `object_tags` — все теги OSM в JSON-формате.  

Эти данные используются для **генерации дополнительных признаков** (например, количества школ, вузов или станций метро рядом с каждым банкоматом), что позволяет улучшить модели прогнозирования популярности и спроса.


In [ ]:
def safe_overpass_request(query, retries=5) -> dict | None:
    """
    Выполняет безопасный запрос к Overpass API с автоматическими повторами,
    переключением между зеркалами и задержками при превышении лимитов.

    Параметры:
    ----------
    query : str
        OverpassQL запрос
    retries : int
        Количество попыток повторного выполнения

    Возвращает:
    -----------
    dict | None :
        JSON-ответ сервера или None в случае неудачи
    """
    # Адреса зеркал Overpass API (на случай недоступности основного)
    OVERPASS_SERVERS = [
        "https://overpass.openstreetmap.fr/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://overpass.osm.ch/api/interpreter"
    ]
    
    for attempt in range(retries):
        # Перебор всех зеркал
        for base_url in OVERPASS_SERVERS:
            try:
                # Отправка запроса
                resp = requests.get(base_url, params={"data": query}, timeout=90)

                # Успешный ответ
                if resp.status_code == 200:
                    return resp.json()

                # Превышен лимит запросов
                elif resp.status_code == 429:
                    wait = (attempt + 1) * 15
                    print(f"HTTP 429. Пауза {wait} сек...")
                    time.sleep(wait)

                # Таймаут шлюза
                elif resp.status_code == 504:
                    print("HTTP 504. Повтор...")
                    time.sleep(10)

                # Неожиданный статус ответа
                else:
                    print(f"Неожиданный ответ {resp.status_code} от {base_url}")

            except requests.exceptions.RequestException as e:
                print(f"Ошибка запроса к {base_url}: {e}")
                time.sleep(5)

        # Все зеркала недоступны — ждём и повторяем
        print("Все зеркала недоступны, ожидание 30 сек...")
        time.sleep(30)
        
    # Все попытки исчерпаны
    print("Все попытки исчерпаны.")
    return None

In [ ]:
def process_osm_dataset(df):
    """
    Обрабатывает набор данных с банкоматами и выполняет поиск объектов инфраструктуры
    вокруг каждого банкомата в радиусе 300 метров.

    Поддерживает:
    • автоматическую дозагрузку с места остановки
    • периодическое сохранение результатов в CSV
    • логирование прогресса

    Параметры:
    ----------
    df : pd.DataFrame
        Таблица с банкоматами (обязательные колонки: id, lat, long)
    """
    # Пути к файлам для данных и прогресса
    output_file = os.path.join("data", "POI_3.csv")
    progress_file = os.path.join("data", "POI_3_progress.json")

    # Загрузка прогресса, если файл существует
    start_index = 0
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            try:
                progress = json.load(f)
                start_index = progress.get("last_index", 0)
                print(f"Возобновление с индекса {start_index}")
            except json.JSONDecodeError:
                pass

    results = []

    # Перебор строк датафрейма с индикатором прогресса
    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df)), start=0):
        if i < start_index:
            continue
        
        # Извлечение координат и ID банкомата
        lat, lon = row.get("geo_lat"), row.get("geo_lon")
        atm_id = row.get("id")

        # Пропуск строк без координат
        if pd.isna(lat) or pd.isna(lon):
            continue

        # Формирование Overpass-запроса
        query = f"""
        [out:json][timeout:60];
        (
          node["name"~"моу|сош|гимназия|лицей|институт|университет|академия|ниу|почта|отделение связи|метро|school|academy|camp"](around:300,{lat},{lon});
          way["name"~"моу|сош|гимназия|лицей|институт|университет|академия|ниу|почта|отделение связи|метро|school|academy|camp"](around:300,{lat},{lon});
          node["amenity"~"school|university|post_office"](around:300,{lat},{lon});
          node["railway"~"subway|station"](around:300,{lat},{lon});
        );
        out center;
        """

        # Выполнение безопасного запроса
        data = safe_overpass_request(query)
        if not data:
            continue
        
        # Обработка элементов ответа
        for el in data.get("elements", []):
            results.append({
                "atm_id": atm_id,
                "object_id": el.get("id"),
                "object_type": el.get("type"),
                "object_lat": el.get("lat") or el.get("center", {}).get("lat"),
                "object_lon": el.get("lon") or el.get("center", {}).get("lon"),
                "object_name": el.get("tags", {}).get("name"),
                "object_tags": json.dumps(el.get("tags", {}), ensure_ascii=False),
            })

        # Промежуточное сохранение каждые 100 объектов
        if (i + 1) % 100 == 0 or i == len(df) - 1:
            if results:
                df_temp = pd.DataFrame(results)
                df_temp.to_csv(output_file, mode="a", header=not os.path.exists(output_file),
                               index=False, encoding="utf-8")
                results.clear()

            # Сохранение индекса прогресса
            with open(progress_file, "w", encoding="utf-8") as f:
                json.dump({"last_index": i + 1}, f)

            print(f"Сохранено: {i + 1} / {len(df)}")

        # Случайная пауза между запросами
        time.sleep(random.uniform(2.5, 5.0))

    # Завершение процесса
    print(f"Завершено. Результаты в {output_file}")


In [ ]:
# Запуск парсинга
process_osm_dataset(clean_data.copy())

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# -----------------------------------------
# 1. Парсинг населения по регионам РФ
# Источник: https://www.ididb.ru/runet/map/
# -----------------------------------------

url_population = "https://www.ididb.ru/runet/map/"
response = requests.get(url_population)
soup = BeautifulSoup(response.text, "html.parser")

population_tables = soup.find_all("table")

data_population = []

for table in population_tables:
    for row in table.find_all("tr")[1:]:
        cols = [col.get_text(strip=True) for col in row.find_all("td")]
        if len(cols) >= 3:
            region = cols[1]
            population = cols[2].replace(" ", "")
            if population.isdigit():
                data_population.append([region, int(population)])

df_population = pd.DataFrame(data_population, columns=["region", "population"])


# -----------------------------------------
# 2. Парсинг площади и плотности населения
# Источник: https://statbase.ru/data/rus-population-by-region-national-stat/
# -----------------------------------------

url_area = "https://statbase.ru/data/rus-population-by-region-national-stat/"
response2 = requests.get(url_area)
soup2 = BeautifulSoup(response2.text, "html.parser")

table2 = soup2.find("table")
rows_area = []
for tr in table2.find_all("tr")[1:]:
    cols = [td.get_text(strip=True) for td in tr.find_all("td")]
    if len(cols) >= 4:
        region = cols[1]
        area_km2 = cols[2].replace(" ", "").replace(",", ".")
        density = cols[3].replace(" ", "").replace(",", ".")
        rows_area.append([region, float(area_km2), float(density)])

df_area = pd.DataFrame(rows_area, columns=["region", "area_km2", "density_per_km2"])


# -----------------------------------------
# 3. Очистка и унификация названий регионов
# -----------------------------------------

def normalize_region_name(name: str) -> str:
    name = name.strip()
    replacements = {
        "Москва": "г. Москва",
        "Санкт-Петербург": "г. Санкт-Петербург",
        "Еврейская АО": "Еврейская автономная область",
        "ХМАО": "Ханты-Мансийский АО",
        "ЯНАО": "Ямало-Ненецкий АО"
    }
    return replacements.get(name, name)

df_population["region"] = df_population["region"].apply(normalize_region_name)
df_area["region"] = df_area["region"].apply(normalize_region_name)


# -----------------------------------------
# 4. Объединение двух источников
# -----------------------------------------

df_final = pd.merge(df_population, df_area, on="region", how="outer")

# сортировка по названию
df_final = df_final.sort_values("region").reset_index(drop=True)

# -----------------------------------------
# 5. Сохранение
# -----------------------------------------

output_path = "regions_population_density_area_2024.csv"
df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Готово. Сохранён файл:", output_path)
print(df_final.head())
print("\nКоличество субъектов РФ в итоговом файле:", len(df_final))